# Plant Biomechanics + Agri-Robot — Modular Entry Point

This notebook is a thin driver over the `plant_sim/` Python package. All simulation logic lives in modules so it can be imported from Isaac Sim, ROS 2 nodes, or unit tests.

**Modules**

| File | Purpose |
|---|---|
| `plant_sim/config.py`     | Geometry, material, weather, integration constants |
| `plant_sim/plant.py`      | Stem + leaf graph builder (Young's-modulus stiffness) |
| `plant_sim/physics.py`    | NVIDIA Warp GPU kernel (gravity, wind drag, rain, soil) |
| `plant_sim/pbd.py`        | Position-Based Dynamics constraint solver |
| `plant_sim/robot.py`      | 3-link arm forward kinematics + soft contact |
| `plant_sim/meshes.py`     | Open3D cylinder / sphere / ground builders |
| `plant_sim/simulation.py` | Interactive visualization loop |

**Install**
```bash
pip install warp-lang numpy scipy matplotlib open3d
```

In [1]:
import warp as wp

wp.init()
print("Device:", wp.get_device())

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 4070" (12 GiB, sm_89, mempool enabled)
   Kernel cache:
     /home/sudharshan/.cache/warp/1.12.0
Device: cuda:0


## 1. Build the plant

`build_single_plant` returns a dict of NumPy arrays describing nodes, edges, rest-lengths, per-edge bending stiffness, and root/leaf flags.

In [5]:
from plant_sim import build_single_plant

plant = build_single_plant(x_offset=0.0)
print(f"Plant: {plant['n_nodes']} nodes, {len(plant['parents'])} edges")

Plant: 32 nodes, 31 edges


## 2. Launch the interactive 3D viewer

Open3D window opens with the plant, a robot arm, and a ground grid. The simulation runs wind + rain + robot contact on the GPU for ~10 seconds (600 frames).

In [7]:
from plant_sim import visualize_plant_and_robot

visualize_plant_and_robot(plant)
print("Done.")

ModuleNotFoundError: No module named 'open3d'

In [10]:
# ══════════════════════════════════════════════════
#  FIX 1+2: Robot arm — base on ground, closer to plant
# ══════════════════════════════════════════════════
class SimpleRobotArm:
    def __init__(self, base_pos=(0.35, 0.0, 0.0)):   # was 0.7 → now 0.35
        self.base = np.array(base_pos, dtype=np.float32)
        # Shorter links so robot looks correctly sized next to plant
        self.link_lengths = [0.45, 0.30, 0.18]

    def forward_kinematics(self, t):
        # Base is FIXED on ground (no vertical motion)
        # A rigid vertical post holds the first joint up
        base_height = 0.45                              # post height

        # Joint 1 sweeps the arm horizontally toward plant
        # Range: -20° to -70° from horizontal (reaching forward-down)
        theta1 = np.deg2rad(-20) - np.deg2rad(50) * (0.5 - 0.5 * np.cos(t * 2 * np.pi))

        # Joint 2 curls to bring gripper down to plant
        theta2 = np.deg2rad(-30) - np.deg2rad(40) * (0.5 - 0.5 * np.cos(t * 2 * np.pi))

        L1, L2, L3 = self.link_lengths

        # p0 = base on ground
        p0 = self.base.copy()
        # p1 = top of vertical post (joint 1)
        p1 = p0 + np.array([0.0, base_height, 0.0])
        # p2 = end of first reaching link (joint 2)
        # Points toward -X (toward plant at x=0)
        p2 = p1 + np.array([-L1 * np.cos(theta1),
                             L1 * np.sin(theta1), 0.0])
        # p3 = end of second link
        theta_abs = theta1 + theta2
        p3 = p2 + np.array([-L2 * np.cos(theta_abs),
                             L2 * np.sin(theta_abs), 0.0])
        # p4 = gripper tip
        theta_grip = theta_abs + np.deg2rad(-20)
        p4 = p3 + np.array([-L3 * np.cos(theta_grip),
                             L3 * np.sin(theta_grip), 0.0])

        return [p0, p1, p2, p3, p4]

    def get_gripper_pos(self, t):
        return self.forward_kinematics(t)[-1]


# ══════════════════════════════════════════════════
#  FIX 3: Ground with floor grid for visibility
# ══════════════════════════════════════════════════
def make_ground_with_grid(size=2.5, color=(0.45, 0.32, 0.20)):
    """Ground plane + grid lines for depth perception."""
    meshes = []

    # Main ground slab
    ground = o3d.geometry.TriangleMesh.create_box(
        width=size, height=0.01, depth=size)
    ground.translate((-size / 2, -0.01, -size / 2))
    ground.paint_uniform_color(color)
    ground.compute_vertex_normals()
    meshes.append(ground)

    # Grid lines on top of ground
    grid_lines = o3d.geometry.LineSet()
    points = []
    lines  = []
    n_lines = 11
    step = size / (n_lines - 1)
    idx = 0
    for i in range(n_lines):
        x = -size / 2 + i * step
        points.append([x, 0.001, -size / 2])
        points.append([x, 0.001,  size / 2])
        lines.append([idx, idx + 1])
        idx += 2
        z = -size / 2 + i * step
        points.append([-size / 2, 0.001, z])
        points.append([ size / 2, 0.001, z])
        lines.append([idx, idx + 1])
        idx += 2
def make_ground_with_grid(size=2.5, color=(0.45, 0.32, 0.20)):
    """Ground plane + grid lines for depth perception."""
    meshes = []

    # Main ground slab
    ground = o3d.geometry.TriangleMesh.create_box(
        wi
    grid_lines.points = o3d.utility.Vector3dVector(points)
    grid_lines.lines  = o3d.utility.Vector2iVector(lines)
    grid_lines.paint_uniform_color([0.3, 0.22, 0.14])
    meshes.append(grid_lines)

    return meshes


# ══════════════════════════════════════════════════
#  FIX 4: Better camera + render settings
#  Replace your visualize_plant_and_robot function with this
# ══════════════════════════════════════════════════
def visualize_plant_and_robot(plant_data):
    n = len(plant_data["pos"])
    pos_np  = plant_data["pos"].copy()
    vel_np  = plant_data["vel"].copy()
    rest_np = plant_data["rest_pos"]

    pos_gpu  = wp.array(pos_np, dtype=wp.vec3,   device="cuda")
    vel_gpu  = wp.array(vel_np, dtype=wp.vec3,   device="cuda")
    mass_gpu = wp.array(plant_data["mass"],         dtype=wp.float32, device="cuda")
    area_gpu = wp.array(plant_data["frontal_area"], dtype=wp.float32, device="cuda")
    root_gpu = wp.array(plant_data["is_root"],      dtype=wp.int32,   device="cuda")
    leaf_gpu = wp.array(plant_data["is_leaf"],      dtype=wp.int32,   device="cuda")

    robot = SimpleRobotArm(base_pos=(0.35, 0.0, 0.0))

    vis = o3d.visualization.Visualizer()
    vis.create_window(
        window_name="Plant Biomechanics + Agri-Robot (GPU Simulation)",
        width=1280, height=720)

    # FIX 3: Better ground with grid
    for g in make_ground_with_grid():
        vis.add_geometry(g)

    # Coordinate axes at origin
    coord = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.12)
    vis.add_geometry(coord)

    # Render options — cleaner look
    ropt = vis.get_render_option()
    ropt.background_color = np.array([0.94, 0.96, 0.98])
    ropt.light_on = True
    ropt.mesh_show_back_face = True

    plant_geoms = []
    robot_geoms = []

    # FIX 4: Better camera — slightly elevated, looking at plant
    ctr = vis.get_view_control()
    ctr.set_lookat([0.2, 0.5, 0.0])
    ctr.set_front([0.2, -0.15, -1.0])       # slightly from above
    ctr.set_up([0.0, 1.0, 0.0])
    ctr.set_zoom(0.45)

    sub_dt     = DT / SUBSTEPS
    frame      = 0
    max_frames = 600                         # ~10 seconds of sim

    print("\n" + "="*60)
    print(" 3D Visualization — close window or Ctrl+C to exit")
    print("="*60)
    print(" Left drag : rotate  |  Right drag : pan  |  Scroll : zoom")
    print("="*60 + "\n")

    try:
        while frame < max_frames:
            # Robot cycles every 150 frames (~2.5 seconds)
            t_robot = (frame % 150) / 150.0
            w       = wind_speed(frame)
            rain_on = 1 if rain_intensity(frame) > 0.0 else 0

            for _ in range(SUBSTEPS):
                wp.launch(k_apply_forces, dim=n,
                          inputs=[pos_gpu, vel_gpu, mass_gpu, area_gpu,
                                  root_gpu, leaf_gpu, w, rain_on,
                                  RHO_AIR, RHO_WATER, CD_STEM, CD_LEAF,
                                  RAIN_RATE, V_RAIN,
                                  DAMPING, ROOT_DAMPING, SOIL_DEPTH, sub_dt])

                pos_np = pos_gpu.numpy().copy()
                vel_np = vel_gpu.numpy().copy()

                if not np.all(np.isfinite(pos_np)):
                    pos_np = rest_np.copy()
                    vel_np = np.zeros_like(pos_np)

                pos_np[0] = [0.0, 0.0, 0.0]
                vel_np[0] = [0.0, 0.0, 0.0]

                gripper = robot.get_gripper_pos(t_robot)
                vel_np = apply_robot_contact(pos_np, vel_np, gripper,
                                              plant_data,
                                              force_strength=60.0)  # stronger push

                pos_np = pbd_solve(pos_np, rest_np,
                                    plant_data["parents"], plant_data["children"],
                                    plant_data["L0"], plant_data["bend_k"],
                                    plant_data["is_root"], n_iters=PBD_ITERS)

                pos_gpu = wp.array(pos_np, dtype=wp.vec3, device="cuda")
                vel_gpu = wp.array(vel_np, dtype=wp.vec3, device="cuda")

            # Remove previous frame geometry
            for g in plant_geoms + robot_geoms:
                vis.remove_geometry(g, reset_bounding_box=False)
            plant_geoms.clear()
            robot_geoms.clear()

            # Stem
            stem_positions = pos_np[:STEM_NODES]
            plant_geoms.extend(make_stem_mesh(stem_positions))

            # Leaves
            cursor = STEM_NODES
            for (attach, ang, n_seg) in LEAF_CONFIG:
                leaf_positions = np.vstack([
                    pos_np[attach:attach+1],
                    pos_np[cursor:cursor + n_seg]
                ])
                plant_geoms.extend(make_leaf_mesh(leaf_positions))
                cursor += n_seg

            # Robot arm
            joints = robot.forward_kinematics(t_robot)
            robot_geoms.extend(make_robot_arm_mesh(joints))

            for g in plant_geoms + robot_geoms:
                vis.add_geometry(g, reset_bounding_box=False)

            vis.poll_events()
            vis.update_renderer()
            frame += 1
            time.sleep(0.016)

    except KeyboardInterrupt:
        print("\nStopped by user.")

    vis.destroy_window()
    print(f"\nCompleted {frame} frames.")

In [15]:
"""
Plant + Robot Arm visualization — fully corrected single-file script.

Install first:
    pip install open3d warp-lang numpy

Controls:
    Left drag  → rotate camera
    Right drag → pan
    Scroll     → zoom
    Ctrl+C     → exit
"""

import open3d as o3d
import numpy as np
import time
import warp as wp

wp.init()
print("Device:", wp.get_device())


# ══════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════
STEM_NODES   = 12
STEM_HEIGHT  = 1.0
L0_STEM      = STEM_HEIGHT / (STEM_NODES - 1)
R_BASE       = 0.008
R_TIP        = 0.002
RHO_STEM     = 700.0
CD_STEM      = 1.2

E_STEM       = 2.0e7
E_LEAF       = 5.0e6

LEAF_CONFIG  = [
    (3,  55, 3), (3, -55, 3),
    (6,  60, 4), (6, -60, 4),
    (9,  50, 3), (9, -50, 3),
]
L0_LEAF      = 0.055
R_LEAF       = 0.0015
A_LEAF_BLADE = 0.005
RHO_LEAF     = 500.0
CD_LEAF      = 2.5

G            = 9.81
RHO_AIR      = 1.225
RHO_WATER    = 1000.0
V_RAIN       = 9.0
RAIN_RATE    = 8.0e-5

SOIL_DEPTH   = 0.05
ROOT_DAMPING = 0.55
DT           = 1.0 / 60.0
SUBSTEPS     = 8
DAMPING      = 0.97
PBD_ITERS    = 4

WIND_CALM   = 0.5
WIND_STRONG = 15.0
RAIN_START  = 50
RAIN_END    = 200


def wind_speed(frame):
    if frame < RAIN_START:      return WIND_STRONG
    if frame < RAIN_END:        return WIND_STRONG * 1.2
    if frame < RAIN_END + 30:   return 3.0
    return WIND_CALM


def rain_intensity(frame):
    if frame < RAIN_START:      return 0.0
    if frame < RAIN_START + 15: return 1.0
    if frame < RAIN_END:        return 2.5
    return 0.0


# ══════════════════════════════════════════════════
#  PHYSICS — Young's modulus based stiffness
# ══════════════════════════════════════════════════
def stem_radius(i, n=STEM_NODES):
    t = i / (n - 1)
    return R_BASE + (R_TIP - R_BASE) * t

def area(r):          return np.pi * r * r
def second_moment(r): return np.pi * r**4 / 4.0

def flexural_rigidity(E, r):
    return E * second_moment(r)

def compute_pbd_stiffness(E, r, L0):
    EI     = flexural_rigidity(E, r)
    EI_ref = flexural_rigidity(E_STEM, R_BASE)
    ratio  = EI / EI_ref
    return 0.04 + (0.92 - 0.04) * np.clip(ratio, 0.0, 1.0) ** 0.4


def build_single_plant(x_offset=0.0):
    pos, mass, is_root, is_leaf, front_area = [], [], [], [], []
    parents, children, L0_list, bend_k = [], [], [], []

    for i in range(STEM_NODES):
        y = i * L0_STEM
        pos.append([x_offset, y, 0.0])
        is_root.append(1 if i == 0 else 0)
        is_leaf.append(0)
        r = stem_radius(i)
        mass.append(max(RHO_STEM * area(r) * L0_STEM, 1e-5))
        front_area.append(2.0 * r * L0_STEM)
        if i > 0:
            r_edge = 0.5 * (stem_radius(i-1) + stem_radius(i))
            parents.append(i - 1); children.append(i)
            L0_list.append(L0_STEM)
            bend_k.append(compute_pbd_stiffness(E_STEM, r_edge, L0_STEM))

    for (attach, angle_deg, n_seg) in LEAF_CONFIG:
        a = np.radians(angle_deg)
        dx, dy = np.sin(a) * L0_LEAF, np.cos(a) * L0_LEAF
        base = pos[attach]; prev = attach
        leaf_stiff = compute_pbd_stiffness(E_LEAF, R_LEAF, L0_LEAF)
        for j in range(1, n_seg + 1):
            idx = len(pos)
            pos.append([base[0] + dx*j, base[1] + dy*j, 0.0])
            is_root.append(0); is_leaf.append(1)
            mass.append(RHO_LEAF * area(R_LEAF) * L0_LEAF)
            front_area.append(A_LEAF_BLADE)
            parents.append(prev); children.append(idx)
            L0_list.append(L0_LEAF)
            bend_k.append(leaf_stiff)
            prev = idx

    pos_np = np.asarray(pos, dtype=np.float32)
    return dict(
        n_nodes=len(pos),
        pos=pos_np,
        rest_pos=pos_np.copy(),
        vel=np.zeros_like(pos_np),
        mass=np.asarray(mass, dtype=np.float32),
        is_root=np.asarray(is_root, dtype=np.int32),
        is_leaf=np.asarray(is_leaf, dtype=np.int32),
        frontal_area=np.asarray(front_area, dtype=np.float32),
        parents=np.asarray(parents, dtype=np.int32),
        children=np.asarray(children, dtype=np.int32),
        L0=np.asarray(L0_list, dtype=np.float32),
        bend_k=np.asarray(bend_k, dtype=np.float32),
    )


# ══════════════════════════════════════════════════
#  GPU KERNEL
# ══════════════════════════════════════════════════
@wp.kernel
def k_apply_forces(positions:  wp.array(dtype=wp.vec3),
                   velocities: wp.array(dtype=wp.vec3),
                   mass:       wp.array(dtype=wp.float32),
                   area_arr:   wp.array(dtype=wp.float32),
                   is_root:    wp.array(dtype=wp.int32),
                   is_leaf:    wp.array(dtype=wp.int32),
                   wind_x:     float, rain_on: int,
                   rho_air:    float, rho_w:   float,
                   cd_stem:    float, cd_leaf:  float,
                   rain_rate:  float, v_rain:   float,
                   damping:    float, soil_damp: float,
                   soil_y:     float, dt:       float):
    i = wp.tid()
    if is_root[i] == 1:
        velocities[i] = wp.vec3(0.0, 0.0, 0.0)
        return
    m = mass[i]
    f = wp.vec3(0.0, -G * m, 0.0)
    v_rel = wp.vec3(wind_x, 0.0, 0.0) - velocities[i]
    speed = wp.length(v_rel)
    cd = cd_stem
    if is_leaf[i] == 1: cd = cd_leaf
    h_factor = 0.3 + 0.8 * positions[i][1]
    f = f + v_rel * (0.5 * rho_air * cd * area_arr[i] * speed * h_factor)
    if rain_on == 1:
        rf = rho_w * rain_rate * v_rain * area_arr[i]
        if is_leaf[i] == 1: rf = rf * 30.0
        else:               rf = rf * 5.0
        f = f + wp.vec3(0.5 * rf, -rf, 0.0)
    a = f / m
    a_mag = wp.length(a)
    if a_mag > 800.0: a = a * (800.0 / a_mag)
    d = damping
    if positions[i][1] < soil_y: d = soil_damp
    velocities[i] = (velocities[i] + a * dt) * d
    positions[i]  =  positions[i]  + velocities[i] * dt
    if positions[i][1] < 0.0:
        positions[i]  = wp.vec3(positions[i][0], 0.0, positions[i][2])
        velocities[i] = wp.vec3(velocities[i][0], 0.0, velocities[i][2])


# ══════════════════════════════════════════════════
#  PBD SOLVER
# ══════════════════════════════════════════════════
def pbd_solve(pos, rest_pos, parents, children, L0, bend_k, is_root, n_iters=4):
    for _ in range(n_iters):
        for e in range(len(parents)):
            p, c = parents[e], children[e]
            diff = pos[c] - pos[p]
            dist = np.linalg.norm(diff)
            if dist > 1e-8:
                n_vec = diff / dist
                err = dist - L0[e]
                if is_root[p]:
                    pos[c] -= n_vec * err
                else:
                    pos[p] += n_vec * err * 0.5
                    pos[c] -= n_vec * err * 0.5
        for e in range(len(parents)):
            p, c = parents[e], children[e]
            rest_dir = rest_pos[c] - rest_pos[p]
            rest_len = np.linalg.norm(rest_dir)
            if rest_len < 1e-8: continue
            rest_dir /= rest_len
            cur_dir = pos[c] - pos[p]
            cur_len = np.linalg.norm(cur_dir)
            if cur_len < 1e-8: continue
            target = pos[p] + rest_dir * cur_len
            pos[c] = pos[c] * (1.0 - bend_k[e]) + target * bend_k[e]
    pos[:, 1] = np.maximum(pos[:, 1], 0.0)
    return pos


# ══════════════════════════════════════════════════
#  ROBOT ARM — base on ground, reaches toward plant
# ══════════════════════════════════════════════════
class SimpleRobotArm:
    def __init__(self, base_pos=(0.35, 0.0, 0.0)):
        self.base = np.array(base_pos, dtype=np.float32)
        self.link_lengths = [0.45, 0.30, 0.18]

    def forward_kinematics(self, t):
        base_height = 0.45

        # Smooth reaching motion: arm extends toward plant and retracts
        s = 0.5 - 0.5 * np.cos(t * 2 * np.pi)  # 0 → 1 → 0

        theta1 = np.deg2rad(-20) - np.deg2rad(50) * s
        theta2 = np.deg2rad(-30) - np.deg2rad(40) * s

        L1, L2, L3 = self.link_lengths

        # p0 = base on ground
        p0 = self.base.copy()
        # p1 = top of rigid vertical post
        p1 = p0 + np.array([0.0, base_height, 0.0])
        # p2 = end of first reaching link (reaches in -X toward plant)
        p2 = p1 + np.array([-L1 * np.cos(theta1),
                             L1 * np.sin(theta1), 0.0])
        # p3 = end of second link
        theta_abs = theta1 + theta2
        p3 = p2 + np.array([-L2 * np.cos(theta_abs),
                             L2 * np.sin(theta_abs), 0.0])
        # p4 = gripper tip
        theta_grip = theta_abs + np.deg2rad(-20)
        p4 = p3 + np.array([-L3 * np.cos(theta_grip),
                             L3 * np.sin(theta_grip), 0.0])

        return [p0, p1, p2, p3, p4]

    def get_gripper_pos(self, t):
        return self.forward_kinematics(t)[-1]


def apply_robot_contact(pos, vel, gripper_pos, field, force_strength=60.0):
    contact_radius = 0.08
    for i in range(len(pos)):
        if field["is_root"][i] == 1:
            continue
        diff = pos[i] - gripper_pos
        dist = np.linalg.norm(diff)
        if dist < contact_radius and dist > 1e-6:
            direction   = diff / dist
            penetration = contact_radius - dist
            push = direction * penetration * force_strength
            vel[i] += push * DT / SUBSTEPS
    return vel


# ══════════════════════════════════════════════════
#  OPEN3D MESH BUILDERS
# ══════════════════════════════════════════════════
def _align_cylinder(cyl, p0, p1):
    vec    = p1 - p0
    length = np.linalg.norm(vec)
    if length < 1e-5:
        return None
    z_axis  = np.array([0.0, 0.0, 1.0])
    dir_vec = vec / length
    axis    = np.cross(z_axis, dir_vec)
    angle   = np.arccos(np.clip(np.dot(z_axis, dir_vec), -1.0, 1.0))
    if np.linalg.norm(axis) > 1e-6:
        axis /= np.linalg.norm(axis)
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)
        cyl.rotate(R, center=(0, 0, 0))
    cyl.translate((p0 + p1) * 0.5)
    return cyl


def make_stem_mesh(positions, color=(0.11, 0.62, 0.46)):
    meshes = []
    for i in range(len(positions) - 1):
        p0 = np.asarray(positions[i],   dtype=np.float32)
        p1 = np.asarray(positions[i+1], dtype=np.float32)
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        t = i / max(len(positions) - 2, 1)
        r = R_BASE + (R_TIP - R_BASE) * t
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=r * 3.5, height=length, resolution=8)
        cyl.paint_uniform_color(color)
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)
    return meshes


def make_leaf_mesh(positions, color=(0.36, 0.79, 0.65)):
    meshes = []
    for i in range(len(positions) - 1):
        p0 = np.asarray(positions[i],   dtype=np.float32)
        p1 = np.asarray(positions[i+1], dtype=np.float32)
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=0.008, height=length, resolution=6)
        cyl.paint_uniform_color(color)
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)
    return meshes


def make_robot_arm_mesh(joints,
                         link_color=(0.3, 0.3, 0.35),
                         joint_color=(0.2, 0.2, 0.25),
                         gripper_color=(0.85, 0.25, 0.15),
                         base_color=(0.4, 0.4, 0.45)):
    meshes = []

    # Solid base platform on the ground
    base_box = o3d.geometry.TriangleMesh.create_box(
        width=0.10, height=0.04, depth=0.10)
    base_box.translate((joints[0][0] - 0.05, 0.0, -0.05))
    base_box.paint_uniform_color(base_color)
    base_box.compute_vertex_normals()
    meshes.append(base_box)

    # Joint spheres
    for p in joints[1:-1]:
        sphere = o3d.geometry.TriangleMesh.create_sphere(
            radius=0.028, resolution=10)
        sphere.paint_uniform_color(joint_color)
        sphere.compute_vertex_normals()
        sphere.translate(p)
        meshes.append(sphere)

    # Gripper sphere (red)
    gripper = o3d.geometry.TriangleMesh.create_sphere(
        radius=0.04, resolution=12)
    gripper.paint_uniform_color(gripper_color)
    gripper.compute_vertex_normals()
    gripper.translate(joints[-1])
    meshes.append(gripper)

    # Link cylinders
    for i in range(len(joints) - 1):
        p0, p1 = joints[i], joints[i+1]
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=0.020, height=length, resolution=10)
        cyl.paint_uniform_color(link_color)
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)

    return meshes


def make_ground_with_grid(size=2.5, color=(0.45, 0.32, 0.20)):
    meshes = []

    ground = o3d.geometry.TriangleMesh.create_box(
        width=size, height=0.01, depth=size)
    ground.translate((-size / 2, -0.01, -size / 2))
    ground.paint_uniform_color(color)
    ground.compute_vertex_normals()
    meshes.append(ground)

    grid_lines = o3d.geometry.LineSet()
    points = []
    lines  = []
    n_lines = 11
    step = size / (n_lines - 1)
    idx = 0
    for i in range(n_lines):
        x = -size / 2 + i * step
        points.append([x, 0.001, -size / 2])
        points.append([x, 0.001,  size / 2])
        lines.append([idx, idx + 1])
        idx += 2
        z = -size / 2 + i * step
        points.append([-size / 2, 0.001, z])
        points.append([ size / 2, 0.001, z])
        lines.append([idx, idx + 1])
        idx += 2

    grid_lines.points = o3d.utility.Vector3dVector(points)
    grid_lines.lines  = o3d.utility.Vector2iVector(lines)
    grid_lines.paint_uniform_color([0.30, 0.22, 0.14])
    meshes.append(grid_lines)

    return meshes


# ══════════════════════════════════════════════════
#  VISUALIZATION LOOP
# ══════════════════════════════════════════════════
def visualize_plant_and_robot(plant_data):
    n = len(plant_data["pos"])
    pos_np  = plant_data["pos"].copy()
    vel_np  = plant_data["vel"].copy()
    rest_np = plant_data["rest_pos"]

    pos_gpu  = wp.array(pos_np, dtype=wp.vec3,   device="cuda")
    vel_gpu  = wp.array(vel_np, dtype=wp.vec3,   device="cuda")
    mass_gpu = wp.array(plant_data["mass"],         dtype=wp.float32, device="cuda")
    area_gpu = wp.array(plant_data["frontal_area"], dtype=wp.float32, device="cuda")
    root_gpu = wp.array(plant_data["is_root"],      dtype=wp.int32,   device="cuda")
    leaf_gpu = wp.array(plant_data["is_leaf"],      dtype=wp.int32,   device="cuda")

    robot = SimpleRobotArm(base_pos=(0.35, 0.0, 0.0))

    vis = o3d.visualization.Visualizer()
    vis.create_window(
        window_name="Plant Biomechanics + Agri-Robot (GPU Simulation)",
        width=1280, height=720)

    for g in make_ground_with_grid():
        vis.add_geometry(g)

    coord = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.12)
    vis.add_geometry(coord)

    ropt = vis.get_render_option()
    ropt.background_color = np.array([0.94, 0.96, 0.98])
    ropt.light_on = True
    ropt.mesh_show_back_face = True

    plant_geoms = []
    robot_geoms = []

    ctr = vis.get_view_control()
    ctr.set_lookat([0.2, 0.5, 0.0])
    ctr.set_front([0.2, -0.15, -1.0])
    ctr.set_up([0.0, 1.0, 0.0])
    ctr.set_zoom(0.45)

    sub_dt     = DT / SUBSTEPS
    frame      = 0
    max_frames = 600

    print("\n" + "="*60)
    print(" 3D Visualization running — close window to exit")
    print("="*60)
    print(" Left drag : rotate  |  Right drag : pan  |  Scroll : zoom")
    print("="*60 + "\n")

    try:
        while frame < max_frames:
            t_robot = (frame % 150) / 150.0
            w       = wind_speed(frame)
            rain_on = 1 if rain_intensity(frame) > 0.0 else 0

            for _ in range(SUBSTEPS):
                wp.launch(k_apply_forces, dim=n,
                          inputs=[pos_gpu, vel_gpu, mass_gpu, area_gpu,
                                  root_gpu, leaf_gpu, w, rain_on,
                                  RHO_AIR, RHO_WATER, CD_STEM, CD_LEAF,
                                  RAIN_RATE, V_RAIN,
                                  DAMPING, ROOT_DAMPING, SOIL_DEPTH, sub_dt])

                pos_np = pos_gpu.numpy().copy()
                vel_np = vel_gpu.numpy().copy()

                if not np.all(np.isfinite(pos_np)):
                    pos_np = rest_np.copy()
                    vel_np = np.zeros_like(pos_np)

                pos_np[0] = [0.0, 0.0, 0.0]
                vel_np[0] = [0.0, 0.0, 0.0]

                gripper = robot.get_gripper_pos(t_robot)
                vel_np = apply_robot_contact(pos_np, vel_np, gripper,
                                              plant_data,
                                              force_strength=60.0)

                pos_np = pbd_solve(pos_np, rest_np,
                                    plant_data["parents"], plant_data["children"],
                                    plant_data["L0"], plant_data["bend_k"],
                                    plant_data["is_root"], n_iters=PBD_ITERS)

                pos_gpu = wp.array(pos_np, dtype=wp.vec3, device="cuda")
                vel_gpu = wp.array(vel_np, dtype=wp.vec3, device="cuda")

            # Remove previous frame's dynamic geometry
            for g in plant_geoms + robot_geoms:
                vis.remove_geometry(g, reset_bounding_box=False)
            plant_geoms.clear()
            robot_geoms.clear()

            # Rebuild stem
            stem_positions = pos_np[:STEM_NODES]
            plant_geoms.extend(make_stem_mesh(stem_positions))

            # Rebuild leaves
            cursor = STEM_NODES
            for (attach, ang, n_seg) in LEAF_CONFIG:
                leaf_positions = np.vstack([
                    pos_np[attach:attach+1],
                    pos_np[cursor:cursor + n_seg]
                ])
                plant_geoms.extend(make_leaf_mesh(leaf_positions))
                cursor += n_seg

            # Rebuild robot arm
            joints = robot.forward_kinematics(t_robot)
            robot_geoms.extend(make_robot_arm_mesh(joints))

            # Add back to scene
            for g in plant_geoms + robot_geoms:
                vis.add_geometry(g, reset_bounding_box=False)

            vis.poll_events()
            vis.update_renderer()
            frame += 1
            time.sleep(0.016)

    except KeyboardInterrupt:
        print("\nStopped by user.")

    vis.destroy_window()
    print(f"\nCompleted {frame} frames of visualization.")


# ══════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════
print("Building plant...")
plant = build_single_plant(x_offset=0.0)
print(f"Plant: {plant['n_nodes']} nodes, {len(plant['parents'])} edges")

print("Launching 3D viewer...")
visualize_plant_and_robot(plant)
print("Done.")

Device: cuda:0
Building plant...
Plant: 32 nodes, 31 edges
Launching 3D viewer...

 3D Visualization running — close window to exit
 Left drag : rotate  |  Right drag : pan  |  Scroll : zoom


Completed 600 frames of visualization.
Done.


In [ ]:
"""
Plant + Robot Arm — INTERACTIVE 3D Scene with Keyboard Controls
─────────────────────────────────────────────────────────────────
Keyboard:
  W  → toggle WIND on/off
  R  → toggle RAIN on/off
  A  → toggle ARM motion on/off
  S  → step wind direction (forward / back / left / right)
  +  → increase wind speed       -  → decrease wind speed
  Q  or Esc → quit

Mouse:
  Left drag  → rotate camera
  Right drag → pan
  Scroll     → zoom
"""

import open3d as o3d
import numpy as np
import time
import warp as wp

wp.init()
print("Device:", wp.get_device())


# ══════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════
STEM_NODES   = 14        # slightly taller stem for sunflower look
STEM_HEIGHT  = 1.2
L0_STEM      = STEM_HEIGHT / (STEM_NODES - 1)
R_BASE       = 0.010
R_TIP        = 0.004
RHO_STEM     = 700.0
CD_STEM      = 1.5

E_STEM       = 1.5e7
E_LEAF       = 3.0e6

# Leaves placed along stem (node_index, angle_deg, num_segments)
LEAF_CONFIG  = [
    (4,   60, 2), (4,  -60, 2),
    (7,   65, 3), (7,  -65, 3),
    (10,  55, 3), (10, -55, 3),
]
L0_LEAF      = 0.07
R_LEAF       = 0.002
A_LEAF_BLADE = 0.008
RHO_LEAF     = 500.0
CD_LEAF      = 2.5

G            = 9.81
RHO_AIR      = 1.225
RHO_WATER    = 1000.0
V_RAIN       = 9.0
RAIN_RATE    = 1.5e-4

SOIL_DEPTH   = 0.05
ROOT_DAMPING = 0.55
DT           = 1.0 / 60.0
SUBSTEPS     = 6
DAMPING      = 0.97
PBD_ITERS    = 3


# ══════════════════════════════════════════════════
#  INTERACTIVE STATE — toggled by keyboard
# ══════════════════════════════════════════════════
class SimState:
    def __init__(self):
        self.wind_on       = True
        self.rain_on       = False
        self.arm_on        = True
        self.wind_speed    = 12.0
        self.wind_dir      = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        self.dir_index     = 0   # 0=+X, 1=-X, 2=+Z, 3=-Z

    def toggle_wind(self):  self.wind_on = not self.wind_on;  self._print()
    def toggle_rain(self):  self.rain_on = not self.rain_on;  self._print()
    def toggle_arm(self):   self.arm_on  = not self.arm_on;   self._print()

    def cycle_wind_direction(self):
        self.dir_index = (self.dir_index + 1) % 4
        dirs = [
            np.array([ 1.0, 0.0, 0.0]),  # +X
            np.array([-1.0, 0.0, 0.0]),  # -X
            np.array([ 0.0, 0.0, 1.0]),  # +Z
            np.array([ 0.0, 0.0, -1.0]), # -Z
        ]
        self.wind_dir = dirs[self.dir_index].astype(np.float32)
        self._print()

    def inc_wind(self):
        self.wind_speed = min(self.wind_speed + 2.0, 30.0)
        self._print()

    def dec_wind(self):
        self.wind_speed = max(self.wind_speed - 2.0, 0.0)
        self._print()

    def _print(self):
        dir_names = ["+X", "-X", "+Z", "-Z"]
        print(f"  Wind: {'ON' if self.wind_on else 'OFF'}  "
              f"speed={self.wind_speed:.1f} m/s  dir={dir_names[self.dir_index]}  |  "
              f"Rain: {'ON' if self.rain_on else 'OFF'}  |  "
              f"Arm: {'ON' if self.arm_on else 'OFF'}")


# ══════════════════════════════════════════════════
#  PHYSICS
# ══════════════════════════════════════════════════
def stem_radius(i, n=STEM_NODES):
    t = i / (n - 1)
    return R_BASE + (R_TIP - R_BASE) * t

def area(r):          return np.pi * r * r
def second_moment(r): return np.pi * r**4 / 4.0
def flexural_rigidity(E, r): return E * second_moment(r)

def compute_pbd_stiffness(E, r, L0):
    EI     = flexural_rigidity(E, r)
    EI_ref = flexural_rigidity(E_STEM, R_BASE)
    ratio  = EI / EI_ref
    return 0.04 + (0.92 - 0.04) * np.clip(ratio, 0.0, 1.0) ** 0.4


def build_sunflower_plant(x_offset=0.0):
    """Build a sunflower: stem + leaves + flower head at top."""
    pos, mass, is_root, is_leaf, is_head, front_area = [], [], [], [], [], []
    parents, children, L0_list, bend_k = [], [], [], []

    # Stem
    for i in range(STEM_NODES):
        y = i * L0_STEM
        pos.append([x_offset, y, 0.0])
        is_root.append(1 if i == 0 else 0)
        is_leaf.append(0)
        is_head.append(1 if i == STEM_NODES - 1 else 0)  # top node is head
        r = stem_radius(i)
        mass.append(max(RHO_STEM * area(r) * L0_STEM, 1e-5))
        # Flower head has extra drag
        fa = 2.0 * r * L0_STEM
        if i == STEM_NODES - 1:
            fa = 0.025     # flower head catches wind like a disk
        front_area.append(fa)
        if i > 0:
            r_edge = 0.5 * (stem_radius(i-1) + stem_radius(i))
            parents.append(i - 1); children.append(i)
            L0_list.append(L0_STEM)
            bend_k.append(compute_pbd_stiffness(E_STEM, r_edge, L0_STEM))

    # Leaves
    for (attach, angle_deg, n_seg) in LEAF_CONFIG:
        a = np.radians(angle_deg)
        dx, dy = np.sin(a) * L0_LEAF, np.cos(a) * L0_LEAF
        base = pos[attach]; prev = attach
        leaf_stiff = compute_pbd_stiffness(E_LEAF, R_LEAF, L0_LEAF)
        for j in range(1, n_seg + 1):
            idx = len(pos)
            pos.append([base[0] + dx*j, base[1] + dy*j, 0.0])
            is_root.append(0); is_leaf.append(1); is_head.append(0)
            mass.append(RHO_LEAF * area(R_LEAF) * L0_LEAF)
            front_area.append(A_LEAF_BLADE)
            parents.append(prev); children.append(idx)
            L0_list.append(L0_LEAF)
            bend_k.append(leaf_stiff)
            prev = idx

    pos_np = np.asarray(pos, dtype=np.float32)
    return dict(
        n_nodes=len(pos),
        pos=pos_np,
        rest_pos=pos_np.copy(),
        vel=np.zeros_like(pos_np),
        mass=np.asarray(mass, dtype=np.float32),
        is_root=np.asarray(is_root, dtype=np.int32),
        is_leaf=np.asarray(is_leaf, dtype=np.int32),
        is_head=np.asarray(is_head, dtype=np.int32),
        frontal_area=np.asarray(front_area, dtype=np.float32),
        parents=np.asarray(parents, dtype=np.int32),
        children=np.asarray(children, dtype=np.int32),
        L0=np.asarray(L0_list, dtype=np.float32),
        bend_k=np.asarray(bend_k, dtype=np.float32),
    )


# ══════════════════════════════════════════════════
#  GPU KERNEL — accepts wind vector (not just X)
# ══════════════════════════════════════════════════
@wp.kernel
def k_apply_forces(positions:  wp.array(dtype=wp.vec3),
                   velocities: wp.array(dtype=wp.vec3),
                   mass:       wp.array(dtype=wp.float32),
                   area_arr:   wp.array(dtype=wp.float32),
                   is_root:    wp.array(dtype=wp.int32),
                   is_leaf:    wp.array(dtype=wp.int32),
                   wind_vec:   wp.vec3, rain_on: int,
                   rho_air:    float, rho_w:   float,
                   cd_stem:    float, cd_leaf:  float,
                   rain_rate:  float, v_rain:   float,
                   damping:    float, soil_damp: float,
                   soil_y:     float, dt:       float):
    i = wp.tid()
    if is_root[i] == 1:
        velocities[i] = wp.vec3(0.0, 0.0, 0.0)
        return
    m = mass[i]
    f = wp.vec3(0.0, -G * m, 0.0)
    v_rel = wind_vec - velocities[i]
    speed = wp.length(v_rel)
    cd = cd_stem
    if is_leaf[i] == 1: cd = cd_leaf
    h_factor = 0.2 + 0.9 * positions[i][1]
    f = f + v_rel * (0.5 * rho_air * cd * area_arr[i] * speed * h_factor)
    if rain_on == 1:
        rf = rho_w * rain_rate * v_rain * area_arr[i]
        if is_leaf[i] == 1: rf = rf * 35.0
        else:               rf = rf * 6.0
        f = f + wp.vec3(0.6 * rf, -rf, 0.0)
    a = f / m
    a_mag = wp.length(a)
    if a_mag > 1000.0: a = a * (1000.0 / a_mag)
    d = damping
    if positions[i][1] < soil_y: d = soil_damp
    velocities[i] = (velocities[i] + a * dt) * d
    positions[i]  =  positions[i]  + velocities[i] * dt
    if positions[i][1] < 0.0:
        positions[i]  = wp.vec3(positions[i][0], 0.0, positions[i][2])
        velocities[i] = wp.vec3(velocities[i][0], 0.0, velocities[i][2])


def pbd_solve(pos, rest_pos, parents, children, L0, bend_k, is_root, n_iters=3):
    for _ in range(n_iters):
        for e in range(len(parents)):
            p, c = parents[e], children[e]
            diff = pos[c] - pos[p]
            dist = np.linalg.norm(diff)
            if dist > 1e-8:
                n_vec = diff / dist
                err = dist - L0[e]
                if is_root[p]:
                    pos[c] -= n_vec * err
                else:
                    pos[p] += n_vec * err * 0.5
                    pos[c] -= n_vec * err * 0.5
        for e in range(len(parents)):
            p, c = parents[e], children[e]
            rest_dir = rest_pos[c] - rest_pos[p]
            rest_len = np.linalg.norm(rest_dir)
            if rest_len < 1e-8: continue
            rest_dir /= rest_len
            cur_dir = pos[c] - pos[p]
            cur_len = np.linalg.norm(cur_dir)
            if cur_len < 1e-8: continue
            target = pos[p] + rest_dir * cur_len
            pos[c] = pos[c] * (1.0 - bend_k[e]) + target * bend_k[e]
    pos[:, 1] = np.maximum(pos[:, 1], 0.0)
    return pos


# ══════════════════════════════════════════════════
#  ROBOT ARM
# ══════════════════════════════════════════════════
class SimpleRobotArm:
    def __init__(self, base_pos=(0.5, 0.0, 0.0)):
        self.base = np.array(base_pos, dtype=np.float32)
        self.post_height = 0.8        # taller to reach flower head
        self.link_lengths = [0.30, 0.25, 0.15]

    def forward_kinematics(self, t):
        s = 0.5 * (1.0 - np.cos(t * 2 * np.pi))
        theta1 = np.deg2rad(30 + 135 * s)
        theta2 = np.deg2rad(-20 - 40 * s)
        L1, L2, L3 = self.link_lengths
        p0 = self.base.copy()
        p1 = p0 + np.array([0.0, self.post_height, 0.0])
        p2 = p1 + np.array([-L1 * np.cos(np.pi - theta1),
                             L1 * np.sin(np.pi - theta1), 0.0])
        theta_abs = (np.pi - theta1) + theta2
        p3 = p2 + np.array([-L2 * np.cos(theta_abs),
                             L2 * np.sin(theta_abs), 0.0])
        theta_grip = theta_abs + np.deg2rad(-15)
        p4 = p3 + np.array([-L3 * np.cos(theta_grip),
                             L3 * np.sin(theta_grip), 0.0])
        return [p0, p1, p2, p3, p4]

    def get_gripper_pos(self, t):
        return self.forward_kinematics(t)[-1]


def apply_robot_contact(pos, vel, gripper_pos, field, force_strength=200.0):
    """
    LOCALIZED contact: only nodes within contact radius push away.
    Closer nodes push harder. This makes the plant deform only
    where the gripper touches it, not the whole stem.
    """
    contact_radius = 0.08
    for i in range(len(pos)):
        if field["is_root"][i] == 1:
            continue
        diff = pos[i] - gripper_pos
        dist = np.linalg.norm(diff)
        if dist < contact_radius and dist > 1e-6:
            direction   = diff / dist
            penetration = contact_radius - dist
            # Quadratic falloff — edge of contact barely affected
            push_magnitude = (penetration / contact_radius) ** 2 * force_strength
            push = direction * push_magnitude
            vel[i] += push * DT / SUBSTEPS
    return vel


# ══════════════════════════════════════════════════
#  RAIN PARTICLES
# ══════════════════════════════════════════════════
class RainParticles:
    def __init__(self, n_drops=300, region=(-1.5, 1.8, -0.8, 0.8)):
        self.n = n_drops
        self.x_min, self.x_max, self.z_min, self.z_max = region
        self.positions = np.zeros((n_drops, 3), dtype=np.float32)
        self.velocities = np.zeros((n_drops, 3), dtype=np.float32)
        self._respawn_all()

    def _respawn_all(self):
        self.positions[:, 0] = np.random.uniform(self.x_min, self.x_max, self.n)
        self.positions[:, 1] = np.random.uniform(0.2, 2.5, self.n)
        self.positions[:, 2] = np.random.uniform(self.z_min, self.z_max, self.n)
        self.velocities[:, 0] = 1.5
        self.velocities[:, 1] = -9.0
        self.velocities[:, 2] = 0.0

    def step(self, dt, active):
        if not active:
            return np.empty((0, 3), dtype=np.float32)
        self.positions += self.velocities * dt
        for i in range(self.n):
            if self.positions[i, 1] < 0.0:
                self.positions[i, 0] = np.random.uniform(self.x_min, self.x_max)
                self.positions[i, 1] = np.random.uniform(1.8, 2.5)
                self.positions[i, 2] = np.random.uniform(self.z_min, self.z_max)
        return self.positions.copy()


# ══════════════════════════════════════════════════
#  OPEN3D MESH BUILDERS
# ══════════════════════════════════════════════════
def _align_cylinder(cyl, p0, p1):
    vec    = p1 - p0
    length = np.linalg.norm(vec)
    if length < 1e-5: return None
    z_axis  = np.array([0.0, 0.0, 1.0])
    dir_vec = vec / length
    axis    = np.cross(z_axis, dir_vec)
    angle   = np.arccos(np.clip(np.dot(z_axis, dir_vec), -1.0, 1.0))
    if np.linalg.norm(axis) > 1e-6:
        axis /= np.linalg.norm(axis)
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)
        cyl.rotate(R, center=(0, 0, 0))
    cyl.translate((p0 + p1) * 0.5)
    return cyl


def make_stem_mesh(positions, color=(0.18, 0.52, 0.28)):
    meshes = []
    for i in range(len(positions) - 1):
        p0 = np.asarray(positions[i],   dtype=np.float32)
        p1 = np.asarray(positions[i+1], dtype=np.float32)
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        t = i / max(len(positions) - 2, 1)
        r = R_BASE + (R_TIP - R_BASE) * t
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=r * 3.0, height=length, resolution=10)
        cyl.paint_uniform_color(color)
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)
    return meshes


def make_leaf_blade(positions, color=(0.28, 0.70, 0.35)):
    """
    Build a broad oval leaf blade instead of thin sticks.
    A leaf is made of small spheres along its length + a central vein.
    """
    meshes = []
    # Central vein (thin cylinder)
    for i in range(len(positions) - 1):
        p0 = np.asarray(positions[i],   dtype=np.float32)
        p1 = np.asarray(positions[i+1], dtype=np.float32)
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=0.004, height=length, resolution=6)
        cyl.paint_uniform_color((0.15, 0.42, 0.20))
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)

    # Leaf blade (flattened spheres along length)
    for i, p in enumerate(positions[1:], start=1):
        t = i / max(len(positions) - 1, 1)
        # Leaf is widest in middle
        width = 0.035 * np.sin(t * np.pi) + 0.010
        blade = o3d.geometry.TriangleMesh.create_sphere(
            radius=width, resolution=10)
        # Flatten it (scale Y and Z)
        flatten = np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.3, 0.0, 0.0],
            [0.0, 0.0, 1.5, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ])
        blade.transform(flatten)
        blade.paint_uniform_color(color)
        blade.compute_vertex_normals()
        blade.translate(p)
        meshes.append(blade)

    return meshes


def make_sunflower_head(center, radius=0.09):
    """Big yellow sunflower with brown center disk."""
    meshes = []
    center = np.asarray(center, dtype=np.float32)

    # Brown center disk
    disk = o3d.geometry.TriangleMesh.create_sphere(
        radius=radius * 0.55, resolution=20)
    flatten = np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 0.3, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ])
    disk.transform(flatten)
    disk.paint_uniform_color((0.45, 0.25, 0.10))
    disk.compute_vertex_normals()
    disk.translate(center)
    meshes.append(disk)

    # Yellow petals around the center
    n_petals = 16
    for k in range(n_petals):
        angle = 2 * np.pi * k / n_petals
        petal = o3d.geometry.TriangleMesh.create_sphere(
            radius=radius * 0.35, resolution=8)
        flatten_petal = np.array([
            [1.0, 0.0, 0.0, 0.0],
            [0.0, 0.25, 0.0, 0.0],
            [0.0, 0.0, 0.6, 0.0],
            [0.0, 0.0, 0.0, 1.0],
        ])
        petal.transform(flatten_petal)
        petal.paint_uniform_color((1.0, 0.75, 0.10))
        petal.compute_vertex_normals()
        # Rotate around Z (petal plane axis)
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(
            np.array([0.0, 0.0, angle]))
        petal.rotate(R, center=(0, 0, 0))
        # Offset outward from disk center
        offset = np.array([
            np.cos(angle) * radius * 0.75,
            np.sin(angle) * radius * 0.75,
            0.0
        ])
        petal.translate(center + offset)
        meshes.append(petal)

    return meshes


def make_robot_arm_mesh(joints,
                         link_color=(0.35, 0.35, 0.40),
                         joint_color=(0.15, 0.15, 0.20),
                         gripper_color=(0.90, 0.20, 0.10),
                         base_color=(0.45, 0.45, 0.50)):
    meshes = []
    base_box = o3d.geometry.TriangleMesh.create_box(
        width=0.14, height=0.06, depth=0.14)
    base_box.translate((joints[0][0] - 0.07, 0.0, -0.07))
    base_box.paint_uniform_color(base_color)
    base_box.compute_vertex_normals()
    meshes.append(base_box)

    for i in range(len(joints) - 1):
        p0, p1 = joints[i], joints[i+1]
        length = np.linalg.norm(p1 - p0)
        if length < 1e-5: continue
        cyl = o3d.geometry.TriangleMesh.create_cylinder(
            radius=0.022, height=length, resolution=10)
        cyl.paint_uniform_color(link_color)
        cyl.compute_vertex_normals()
        _align_cylinder(cyl, p0, p1)
        meshes.append(cyl)

    for p in joints[1:-1]:
        sphere = o3d.geometry.TriangleMesh.create_sphere(
            radius=0.030, resolution=12)
        sphere.paint_uniform_color(joint_color)
        sphere.compute_vertex_normals()
        sphere.translate(p)
        meshes.append(sphere)

    gripper = o3d.geometry.TriangleMesh.create_sphere(
        radius=0.055, resolution=16)
    gripper.paint_uniform_color(gripper_color)
    gripper.compute_vertex_normals()
    gripper.translate(joints[-1])
    meshes.append(gripper)

    return meshes


def make_ground_with_grid(size=3.0, color=(0.45, 0.32, 0.20)):
    meshes = []
    ground = o3d.geometry.TriangleMesh.create_box(
        width=size, height=0.01, depth=size)
    ground.translate((-size / 2, -0.01, -size / 2))
    ground.paint_uniform_color(color)
    ground.compute_vertex_normals()
    meshes.append(ground)

    grid_lines = o3d.geometry.LineSet()
    points = []; lines = []
    n_lines = 13
    step = size / (n_lines - 1)
    idx = 0
    for i in range(n_lines):
        x = -size / 2 + i * step
        points.append([x, 0.002, -size / 2])
        points.append([x, 0.002,  size / 2])
        lines.append([idx, idx + 1]); idx += 2
        z = -size / 2 + i * step
        points.append([-size / 2, 0.002, z])
        points.append([ size / 2, 0.002, z])
        lines.append([idx, idx + 1]); idx += 2

    grid_lines.points = o3d.utility.Vector3dVector(points)
    grid_lines.lines  = o3d.utility.Vector2iVector(lines)
    grid_lines.paint_uniform_color([0.28, 0.20, 0.12])
    meshes.append(grid_lines)
    return meshes


def make_rain_lines(rain_positions):
    if len(rain_positions) == 0:
        return o3d.geometry.LineSet()
    n = len(rain_positions)
    points = np.zeros((2 * n, 3), dtype=np.float32)
    lines = []
    for i, p in enumerate(rain_positions):
        points[2*i]     = p
        points[2*i + 1] = p - np.array([0.04, 0.20, 0.0])
        lines.append([2*i, 2*i + 1])
    ls = o3d.geometry.LineSet()
    ls.points = o3d.utility.Vector3dVector(points)
    ls.lines  = o3d.utility.Vector2iVector(lines)
    ls.paint_uniform_color([0.22, 0.54, 0.86])
    return ls


def make_wind_arrow(direction, strength, position=(-0.6, 1.9, 0.0)):
    """
    Big red arrow in the sky showing wind direction + magnitude.
    Arrow length scales with wind strength.
    """
    meshes = []
    length = 0.08 + 0.04 * strength / 10.0   # 8 cm to ~20 cm
    position = np.asarray(position, dtype=np.float32)
    direction = np.asarray(direction, dtype=np.float32)
    direction = direction / (np.linalg.norm(direction) + 1e-9)

    # Arrow shaft
    shaft = o3d.geometry.TriangleMesh.create_cylinder(
        radius=0.025, height=length, resolution=12)
    shaft.paint_uniform_color((0.90, 0.20, 0.15))
    shaft.compute_vertex_normals()
    _align_cylinder(shaft, position, position + direction * length)
    meshes.append(shaft)

    # Arrow head (cone)
    head = o3d.geometry.TriangleMesh.create_cone(
        radius=0.05, height=0.08, resolution=12)
    head.paint_uniform_color((0.80, 0.10, 0.05))
    head.compute_vertex_normals()
    # Orient the cone
    z_axis  = np.array([0.0, 0.0, 1.0])
    axis    = np.cross(z_axis, direction)
    angle   = np.arccos(np.clip(np.dot(z_axis, direction), -1.0, 1.0))
    if np.linalg.norm(axis) > 1e-6:
        axis /= np.linalg.norm(axis)
        R = o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)
        head.rotate(R, center=(0, 0, 0))
    head.translate(position + direction * length)
    meshes.append(head)

    return meshes


# ══════════════════════════════════════════════════
#  MAIN INTERACTIVE VIEWER
# ══════════════════════════════════════════════════
def visualize_interactive(plant_data):
    n = len(plant_data["pos"])
    pos_np  = plant_data["pos"].copy()
    vel_np  = plant_data["vel"].copy()
    rest_np = plant_data["rest_pos"]

    pos_gpu  = wp.array(pos_np, dtype=wp.vec3,   device="cuda")
    vel_gpu  = wp.array(vel_np, dtype=wp.vec3,   device="cuda")
    mass_gpu = wp.array(plant_data["mass"],         dtype=wp.float32, device="cuda")
    area_gpu = wp.array(plant_data["frontal_area"], dtype=wp.float32, device="cuda")
    root_gpu = wp.array(plant_data["is_root"],      dtype=wp.int32,   device="cuda")
    leaf_gpu = wp.array(plant_data["is_leaf"],      dtype=wp.int32,   device="cuda")

    robot = SimpleRobotArm(base_pos=(0.5, 0.0, 0.0))
    rain  = RainParticles(n_drops=300)
    state = SimState()

    # Use VisualizerWithKeyCallback for interactivity
    vis = o3d.visualization.VisualizerWithKeyCallback()
    vis.create_window(
        window_name="Sunflower Biomechanics + Agri-Robot (INTERACTIVE)",
        width=1280, height=720)

    # Register keyboard callbacks
    def cb_wind(v):  state.toggle_wind();           return False
    def cb_rain(v):  state.toggle_rain();           return False
    def cb_arm(v):   state.toggle_arm();            return False
    def cb_dir(v):   state.cycle_wind_direction();  return False
    def cb_inc(v):   state.inc_wind();              return False
    def cb_dec(v):   state.dec_wind();              return False

    vis.register_key_callback(ord("W"), cb_wind)
    vis.register_key_callback(ord("R"), cb_rain)
    vis.register_key_callback(ord("A"), cb_arm)
    vis.register_key_callback(ord("S"), cb_dir)
    vis.register_key_callback(ord("="), cb_inc)   # + key (unshifted)
    vis.register_key_callback(ord("+"), cb_inc)
    vis.register_key_callback(ord("-"), cb_dec)

    # Static scene
    for g in make_ground_with_grid(size=3.0):
        vis.add_geometry(g)
    coord = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.15)
    vis.add_geometry(coord)

    ropt = vis.get_render_option()
    ropt.background_color = np.array([0.82, 0.91, 0.97])  # sky blue
    ropt.light_on = True
    ropt.mesh_show_back_face = True
    ropt.line_width = 2.0

    plant_geoms = []
    robot_geoms = []
    rain_geom   = None
    arrow_geoms = []
    head_geoms  = []

    ctr = vis.get_view_control()
    ctr.set_lookat([0.15, 0.65, 0.0])
    ctr.set_front([0.3, -0.1, -1.0])
    ctr.set_up([0.0, 1.0, 0.0])
    ctr.set_zoom(0.45)

    sub_dt = DT / SUBSTEPS
    frame  = 0

    print("\n" + "="*66)
    print(" INTERACTIVE 3D SUNFLOWER + AGRI-ROBOT SIMULATION")
    print("="*66)
    print("  W : toggle WIND        R : toggle RAIN        A : toggle ARM")
    print("  S : cycle WIND DIR     + : faster wind        - : slower wind")
    print("  Mouse: rotate / pan / zoom      Close window to exit")
    print("="*66 + "\n")
    state._print()

    try:
        while True:
            t_robot = (frame % 180) / 180.0 if state.arm_on else 0.0
            wind_vec_np = state.wind_dir * (state.wind_speed if state.wind_on else 0.0)
            wind_vec = wp.vec3(float(wind_vec_np[0]),
                                float(wind_vec_np[1]),
                                float(wind_vec_np[2]))
            rain_flag = 1 if state.rain_on else 0

            # ── Physics step ──
            for _ in range(SUBSTEPS):
                wp.launch(k_apply_forces, dim=n,
                          inputs=[pos_gpu, vel_gpu, mass_gpu, area_gpu,
                                  root_gpu, leaf_gpu, wind_vec, rain_flag,
                                  RHO_AIR, RHO_WATER, CD_STEM, CD_LEAF,
                                  RAIN_RATE, V_RAIN,
                                  DAMPING, ROOT_DAMPING, SOIL_DEPTH, sub_dt])

                pos_np = pos_gpu.numpy().copy()
                vel_np = vel_gpu.numpy().copy()

                if not np.all(np.isfinite(pos_np)):
                    pos_np = rest_np.copy()
                    vel_np = np.zeros_like(pos_np)

                pos_np[0] = [0.0, 0.0, 0.0]
                vel_np[0] = [0.0, 0.0, 0.0]

                if state.arm_on:
                    gripper = robot.get_gripper_pos(t_robot)
                    vel_np = apply_robot_contact(pos_np, vel_np, gripper,
                                                  plant_data,
                                                  force_strength=200.0)

                pos_np = pbd_solve(pos_np, rest_np,
                                    plant_data["parents"], plant_data["children"],
                                    plant_data["L0"], plant_data["bend_k"],
                                    plant_data["is_root"], n_iters=PBD_ITERS)

                pos_gpu = wp.array(pos_np, dtype=wp.vec3, device="cuda")
                vel_gpu = wp.array(vel_np, dtype=wp.vec3, device="cuda")

            rain_positions = rain.step(DT, state.rain_on)

            # ── Remove old geometry ──
            for g in plant_geoms + robot_geoms + arrow_geoms + head_geoms:
                vis.remove_geometry(g, reset_bounding_box=False)
            plant_geoms.clear()
            robot_geoms.clear()
            arrow_geoms.clear()
            head_geoms.clear()
            if rain_geom is not None:
                vis.remove_geometry(rain_geom, reset_bounding_box=False)
                rain_geom = None

            # ── Rebuild stem ──
            stem_positions = pos_np[:STEM_NODES]
            plant_geoms.extend(make_stem_mesh(stem_positions))

            # ── Rebuild sunflower head ──
            head_pos = pos_np[STEM_NODES - 1]
            head_geoms.extend(make_sunflower_head(head_pos, radius=0.09))

            # ── Rebuild leaves as blades ──
            cursor = STEM_NODES
            for (attach, ang, n_seg) in LEAF_CONFIG:
                leaf_positions = np.vstack([
                    pos_np[attach:attach+1],
                    pos_np[cursor:cursor + n_seg]
                ])
                plant_geoms.extend(make_leaf_blade(leaf_positions))
                cursor += n_seg

            # ── Robot arm (only draw if active) ──
            if state.arm_on:
                joints = robot.forward_kinematics(t_robot)
                robot_geoms.extend(make_robot_arm_mesh(joints))

            # ── Wind arrow ──
            if state.wind_on and state.wind_speed > 0:
                arrow_geoms.extend(
                    make_wind_arrow(state.wind_dir, state.wind_speed,
                                      position=(-0.6, 1.9, 0.0)))

            # ── Rain ──
            if len(rain_positions) > 0:
                rain_geom = make_rain_lines(rain_positions)
                vis.add_geometry(rain_geom, reset_bounding_box=False)

            # ── Add all back ──
            for g in plant_geoms + robot_geoms + arrow_geoms + head_geoms:
                vis.add_geometry(g, reset_bounding_box=False)

            if not vis.poll_events():
                break
            vis.update_renderer()
            frame += 1
            time.sleep(0.016)

    except KeyboardInterrupt:
        print("\nStopped by user.")

    vis.destroy_window()
    print(f"\nCompleted {frame} frames.")


# ══════════════════════════════════════════════════
#  RUN
# ══════════════════════════════════════════════════
print("Building sunflower...")
plant = build_sunflower_plant(x_offset=0.0)
print(f"Sunflower: {plant['n_nodes']} nodes, {len(plant['parents'])} edges")

print("Launching interactive 3D viewer...")
visualize_interactive(plant)
print("Done.")

Device: cuda:0
Building sunflower...
Sunflower: 30 nodes, 29 edges
Launching interactive 3D viewer...

 INTERACTIVE 3D SUNFLOWER + AGRI-ROBOT SIMULATION
  W : toggle WIND        R : toggle RAIN        A : toggle ARM
  S : cycle WIND DIR     + : faster wind        - : slower wind
  Mouse: rotate / pan / zoom      Close window to exit

  Wind: ON  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
Module __main__ c3f7dbb load on device 'cuda:0' took 218.73 ms  (compiled)
  Wind: OFF  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: ON  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: OFF  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: ON  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: OFF  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: ON  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: OFF  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: ON  speed=12.0 m/s  dir=+X  |  Rain: OFF  |  Arm: ON
  Wind: OFF  speed=12.0 m/s  dir=+X 